In [2]:
import csv

def subtractor_oracle(a2, a1, a0, b2, b1, b0, din):
    """
    Oracle function: computes the expected subtraction result directly in Python.
    Logic: A - B - Din.
    If the result is negative, it means a borrow-out (D2) occurred.
    """
    # Convert 3-bit binary to decimal
    A = (a2 << 2) | (a1 << 1) | a0
    B = (b2 << 2) | (b1 << 1) | b0

    # Perform mathematical subtraction
    # Diff = A - B - Din
    diff = A - B - din

    # Check for borrow out (D2)
    # If diff < 0, it means A was smaller than (B + Din), so we need to borrow.
    if diff < 0:
        d2 = 1
        # To get the 3-bit binary representation of a negative number in this context,
        # we add 8 (which is 2^3) to simulate the borrow from the next non-existent bit.
        diff = diff + 8
    else:
        d2 = 0

    # Extract the decimal result back to 3-bit binary (S2, S1, S0)
    s0 = diff & 1
    s1 = (diff >> 1) & 1
    s2 = (diff >> 2) & 1

    return s2, s1, s0, d2

def validate_truth_table(csv_filename):
    print(f"Starting truth table validation: {csv_filename} ...\n")

    try:
        with open(csv_filename, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)

            # Clean up potential spaces in column names
            fieldnames = [name.strip() for name in reader.fieldnames]
            reader.fieldnames = fieldnames

            row_count = 0

            for row in reader:
                row_count += 1

                # Read current input vector
                a0 = int(row['A0'].strip())
                a1 = int(row['A1'].strip())
                a2 = int(row['A2'].strip())
                b0 = int(row['B0'].strip())
                b1 = int(row['B1'].strip())
                b2 = int(row['B2'].strip())
                din = int(row['Din'].strip())

                # Read actual outputs generated by BENCH simulation
                actual_s0 = int(row['S0'].strip())
                actual_s1 = int(row['S1'].strip())
                actual_s2 = int(row['S2'].strip())
                actual_d2 = int(row['D2'].strip())

                # Compute expected outputs using the Oracle
                exp_s2, exp_s1, exp_s0, exp_d2 = subtractor_oracle(a2, a1, a0, b2, b1, b0, din)

                # Compare actual outputs with expected outputs
                if (actual_s0 != exp_s0) or (actual_s1 != exp_s1) or \
                   (actual_s2 != exp_s2) or (actual_d2 != exp_d2):

                    print("-" * 65)
                    print("Mismatch found!")
                    print(f"First failing input vector occurred at row {row_count}:")
                    print(f"Input vector (A2 A1 A0, B2 B1 B0, Din): {a2}{a1}{a0}, {b2}{b1}{b0}, {din} (Decimal: {a2<<2 | a1<<1 | a0} - {b2<<2 | b1<<1 | b0} - {din})")
                    print(f"Expected output (Oracle S2 S1 S0, D2) : {exp_s2} {exp_s1} {exp_s0}, {exp_d2}")
                    print(f"Actual output   (BENCH  S2 S1 S0, D2) : {actual_s2} {actual_s1} {actual_s0}, {actual_d2}")
                    print("-" * 65)
                    print("\nLikely cause analysis:")
                    print("1. Subtractor Logic Error: The CNF equations generated by the LLM for the 3-bit subtractor are flawed. LLMs often struggle to cascade borrow logic correctly across multiple bits.")
                    print("2. Minterm/Polarity Issue: Check if the borrow-in (Din) and intermediate borrows (D0, D1) are properly connected. A mismatch indicates the synthesized Verilog does not represent true mathematical subtraction.")

                    return # Stop at the first failure as required

            print("Validation passed! All rows match the oracle.")

    except FileNotFoundError:
        print(f"Error: File '{csv_filename}' not found. Please check the path and filename.")
    except KeyError as e:
        print(f"Error: Missing column in CSV file - {e}. Please check your CSV headers. Ensure they are exactly: A0,A1,A2,B0,B1,B2,Din,S0,S1,S2,D2")

# Run the validation
# Make sure to replace 'subtractor_3-bit_typ_tab.csv' with your actual filename
validate_truth_table('subtractor_3-bit_typ_tab.csv')

Starting truth table validation: subtractor_3-bit_typ_tab.csv ...

Validation passed! All rows match the oracle.
